In [1]:
import torch
import transformers as tf
import plotly as pt
import numpy as np
import sklearn as ml

In [2]:
model_name  = "google/gemma-3-1b-pt" 
cfg = tf.AutoConfig.from_pretrained(model_name)
model = tf.AutoModelForCausalLM.from_pretrained(
    model_name,
    config=cfg,
    attn_implementation="eager",
    torch_dtype=torch.float32,
)
model.to("cpu")
tokenizer = tf.AutoTokenizer.from_pretrained(model_name)

/opt/anaconda3/envs/scipy/lib/python3.11/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/opt/anaconda3/envs/scipy/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <EB3FF92A-5EB1-3EE8-AF8B-5923C1265422> /opt/anaconda3/envs/scipy/lib/python3.11/site-packages/torchvision/image.so
  Reason: tried: '/opt/anaconda3/envs/scipy/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/anaconda3/envs/scipy/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/anaconda3/envs/scipy/lib/python3.11/lib-dynload/../../libjpeg.9.dylib' (no such file), '/opt/anaconda3/envs/scipy/bin/../lib/libjpeg.9.dylib' (no such file)'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `lib

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

In [3]:
config = {}
config["LAYERS"] = getattr(cfg, 'num_hidden_layers', 'N/A')
config["HIDDEN_SIZE"] = getattr(cfg, 'hidden_size', 'N/A')
config["ATTENTION_HEADS"] = getattr(cfg, 'num_attention_heads', 'N/A')
config["KV_HEADS"] = getattr(cfg, 'num_key_value_heads', 'N/A')
config["SLIDING_WINDOW"] = getattr(cfg, 'sliding_window', 'N/A')
config['FFN'] = getattr(cfg, 'intermediate_size', 'N/A')
config['VOCAB_SIZE'] = getattr(cfg, 'vocab_size', 'N/A')
config['FINAL_LOGIT_SOFTCAP'] = getattr(cfg, 'final_logit_softcapping', 'N/A')
config['ATTN_LOGIT_SOFTCAP'] = getattr(cfg, 'attn_logit_softcapping', 'N/A')
for name, module in model.named_modules():
    config[f"{name}"] = type(module).__name__

In [4]:
prompt = """Capital of
France is"""
inputs = tokenizer(prompt, return_tensors="pt")
print(inputs)
print(" ".join([tokenizer.convert_ids_to_tokens(i) for i in inputs.input_ids[0].tolist()]))

{'input_ids': tensor([[    2, 64753,   529,   107, 31756,   563]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}
<bos> Capital ▁of 
 France ▁is


In [5]:
def token_label_helper(id_):
    label = tokenizer.convert_ids_to_tokens(id_)
    label = label.replace("▁", "·")
    label = label.replace("\n", "⏎")
    label = label.replace("\t", "⇥")
    return label
def prompt_token_labels(inputs_tnsr):
    # inputs = inputs_tnsr[0].tolist()
    labels = [token_label_helper(i) for i in inputs_tnsr]
    if len(labels) != len(inputs_tnsr):
        raise AssertionError("Length of labels and tokens do not match")
    return ([label if len(label) < 10 else label[:4] + "…" for label in labels],labels)
    

In [6]:
import plotly.graph_objects as go
import plotly.io as pio
fig = go.Figure()

scales = {'sequential': 'Viridis', 'diverging': 'RdBu', 'categorical': ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3', '#FF6692', '#B6E880']}
# Define your custom styling

gemma_template ={
        "layout": {
            "font": {"family": "Arial", "size": 12, "color": "#333"},
            "paper_bgcolor": "#f8f9fa",
            "plot_bgcolor": "#ffffff",
            "title": {"font": {"size": 18, "color": "#000"}},
            "xaxis": {"gridcolor": "#e0e0e0"},
            "yaxis": {"gridcolor": "#e0e0e0"},
        }
    }
pio.templates['gemma_template'] = gemma_template

In [7]:
def heatmap_fn(z, x_labels, y_labels):
    # Get unique x_labels while preserving order
    # unique_x = list(dict.fromkeys(x_labels))
    # print("Test", unique_x)
    fig = go.Figure(data=go.Heatmap(
        z=z,
        x=list(range(len(x_labels))),
        y=y_labels,
        colorscale=scales['sequential'],
        colorbar=dict(title="Attention"),
    ))
    
    fig.update_layout(
        xaxis=dict(
            tickmode='array',
            tickvals=list(range(len(x_labels))),
            ticktext=x_labels
        ),
        template='gemma_template'
    )
    return fig

In [8]:
config

{'LAYERS': 26,
 'HIDDEN_SIZE': 1152,
 'ATTENTION_HEADS': 4,
 'KV_HEADS': 1,
 'SLIDING_WINDOW': 512,
 'FFN': 6912,
 'VOCAB_SIZE': 262144,
 'FINAL_LOGIT_SOFTCAP': None,
 'ATTN_LOGIT_SOFTCAP': None,
 '': 'Gemma3ForCausalLM',
 'model': 'Gemma3TextModel',
 'model.embed_tokens': 'Gemma3TextScaledWordEmbedding',
 'model.layers': 'ModuleList',
 'model.layers.0': 'Gemma3DecoderLayer',
 'model.layers.0.self_attn': 'Gemma3Attention',
 'model.layers.0.self_attn.q_proj': 'Linear',
 'model.layers.0.self_attn.k_proj': 'Linear',
 'model.layers.0.self_attn.v_proj': 'Linear',
 'model.layers.0.self_attn.o_proj': 'Linear',
 'model.layers.0.self_attn.q_norm': 'Gemma3RMSNorm',
 'model.layers.0.self_attn.k_norm': 'Gemma3RMSNorm',
 'model.layers.0.mlp': 'Gemma3MLP',
 'model.layers.0.mlp.gate_proj': 'Linear',
 'model.layers.0.mlp.up_proj': 'Linear',
 'model.layers.0.mlp.down_proj': 'Linear',
 'model.layers.0.mlp.act_fn': 'GELUTanh',
 'model.layers.0.input_layernorm': 'Gemma3RMSNorm',
 'model.layers.0.post_atte

In [17]:
outputs = {}
inputs = {}
def get_layer_hook(layer_num):
    def hook(model, input, output):
        outputs[layer_num] = output.detach().cpu().numpy()
        inputs[layer_num] = input[0].detach().cpu().numpy()

    return hook

model.model.layers[13].post_feedforward_layernorm.register_forward_hook(get_layer_hook('model.layers.13.pre_feedforward_layernorm'))
# Forward pass
inputs_prompt = tokenizer("Hello world", return_tensors="pt")
with torch.no_grad():
    model_outputs = model(**inputs_prompt)

# Access intermediate outputs
print("Out -->",outputs)  # [batch, seq_len, hidden_dim]
print("In -->",inputs)   # [batch, seq_len, hidden_dim]

Out --> {'model.layers.13.pre_feedforward_layernorm': array([[[ -6.520651  ,  -2.5587194 ,  -8.880919  , ...,   3.4068956 ,
          -1.8125802 ,  -2.498098  ],
        [-14.447106  , -27.523693  ,  13.775201  , ...,  25.016685  ,
           4.218295  ,   5.329573  ],
        [-32.279194  ,  -9.599413  ,  -0.23215635, ..., -31.423882  ,
          -1.0199637 ,  -7.069101  ]]], shape=(1, 3, 1152), dtype=float32)}
In --> {'model.layers.13.pre_feedforward_layernorm': array([[[-0.00042528, -0.00025612, -0.00054471, ...,  0.00013641,
         -0.0008294 , -0.00042113],
        [-0.03045195, -0.08903703,  0.0273059 , ...,  0.03237082,
          0.06238107,  0.02903707],
        [-0.05909934, -0.02697332, -0.00039973, ..., -0.03531909,
         -0.01310166, -0.03345417]]], shape=(1, 3, 1152), dtype=float32)}


In [10]:
# keys <class 'torch.Tensor'>
# Out --> {'model.layers.12.pre_feedforward_layernorm': array([[[  6.8228536 ,   2.0604973 , -14.248279  , ...,  -4.1923246 ,
#           -0.57437414,   2.5157247 ],
#         [ 15.561513  ,   1.1429639 , -17.794533  , ...,  -8.807057  ,
#           -0.2044834 ,   0.37308478],
#         [ -9.4363    ,   3.3407087 ,  -6.4893    , ...,  22.475046  ,
#           -1.7454196 ,  -0.25911778]]], shape=(1, 3, 1152), dtype=float32)}
# In --> {'model.layers.12.pre_feedforward_layernorm': array([[[ 0.00385035,  0.00161424, -0.00704561, ..., -0.00110782,
#          -0.00149993,  0.00221399],
#         [ 0.4765784 ,  0.0485935 , -0.4775194 , ..., -0.12629752,
#          -0.02897893,  0.01781835],
#         [-0.10071582,  0.04949914, -0.0606898 , ...,  0.11232541,
#          -0.08620603, -0.00431291]]], shape=(1, 3, 1152), dtype=float32)}

In [11]:
model.model.layers[10].post_feedforward_layernorm

Gemma3RMSNorm((1152,), eps=1e-06)

# Appendix : Tests

In [12]:
prompt = "Capital of Ethiopia is"
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

Capital of Ethiopia is the city of Addis Ababa. Addis Ababa is the most populous city in the country of Ethiopia and the capital city. Addis Ababa is well known for it’s historical sites and landmarks. It is often referred to as the “the cultural capital of the


In [13]:
from test_phase1 import check_1_1,check_1_2,check_1_3,check_1_4,check_1_5,check_1_6,check_1_7,check_1_8
check_1_1(tokenizer, token_label_helper)



-- fact: 'The capital of France is'
pos       id  raw piece               full label              
  0        2  '<bos>'                 <bos>                   
  1      818  'The'                   The                     
  2     5279  '▁capital'              ·capital                
  3      529  '▁of'                   ·of                     
  4     7001  '▁France'               ·France                 
  5      563  '▁is'                   ·is                     

-- code: 'def my_func(x):\n    y = obj.__init__(x)\n\treturn y_value'
pos       id  raw piece               full label              
  0        2  '<bos>'                 <bos>                   
  1     2063  'def'                   def                     
  2     1041  '▁my'                   ·my                     
  3   236779  '_'                     _                       
  4     6823  'func'                  func                    
  5   236769  '('                     (                       
  6   2367

True

In [14]:
check_1_2("gemma_template",scales)


Task 1.2 - plot theme and colour scales
----------------------------------------------------------------------------
PASS  template 'gemma_template' is registered in plotly.io.templates
NOTE  template is not the default
        -> every figure will need template=... - consider plotly.io.templates.default
PASS  template sets a background colour
PASS  template sets a font family
NOTE  template sets no margins
        -> fine, but margins keep all five plots aligned
PASS  scales has 'sequential', 'diverging', 'categorical'
PASS  sequential: lightness changes in one direction only
PASS  sequential: enough lightness range
PASS  diverging: midpoint is neutral (grey-ish)
PASS  diverging: midpoint is the lightest or darkest point
PASS  diverging: both ends equally strong
PASS  diverging: the two ends are clearly different colours
PASS  categorical: a list of '#RRGGBB' / 'rgb(...)' strings
PASS  categorical: at least 8 colours (token classes in viz 5)
PASS  categorical: every pair of colours i

True

In [15]:
check_1_3(heatmap_fn)


Task 1.3 - numeric-axis heatmap
----------------------------------------------------------------------------
PASS  returns a plotly Figure
PASS  figure contains a heatmap trace
PASS  z is plotted as rows x positions (not transposed)
PASS  x positions are numbers, not token strings
PASS  'a b a b' gives 4 columns, not 2
PASS  x axis is not categorical
PASS  tick text shows the token labels
PASS  tick positions line up with the columns
----------------------------------------------------------------------------
Task 1.3: all checks pass - done.


True

In [16]:
# import torch

# # 1. Is Metal available?
# print(torch.backends.mps.is_available())  # True/False for Metal

# # 2. Is it built into PyTorch?
# print(torch.backends.mps.is_built())  # True/False

# # 3. Check detailed info
# print(f"MPS available: {torch.backends.mps.is_available()}")
# print(f"MPS built: {torch.backends.mps.is_built()}")

# # 4. Move tensor to Metal GPU
# if torch.backends.mps.is_available():
#     x = torch.randn(10, 10).to("mps")
#     print(x.device)  # torch.device('mps:0')
# else:
#     print("Metal GPU not available, falling back to CPU")

# # 5. Check what device you're on
# device = "mps" if torch.backends.mps.is_available() else "cpu"
# print(f"Using device: {device}")